# 1장 2강: 핵심 모델과 선택 기준 압축 정리

## 실습 목표

- Ames Housing의 `SalePrice` 예측 상황에서 **Linear Regression**이 어떤 역할을 하는지 확인합니다.
- KNN과 K-means가 **거리 기반**이라는 점과 스케일에 민감하다는 특징을 구분합니다.
- Decision Tree의 규칙 기반 분할 특징과 스케일링이 필요 없다는 점을 확인합니다.
- K-means와 PCA의 목적이 각각 **군집화 / 차원축소**임을 구분합니다.
- 데이터의 목적과 특성에 따라 다섯 모델 중 **1차 후보를 선택하는 기준**을 정리합니다.

> 이번 실습은 1장 2강 교안에서 다룬 모델의 **핵심 성격과 선택 기준**만 사용합니다.  
> 교안에서 다루지 않은 성능지표, 교차검증, 하이퍼파라미터 탐색, 앙상블 모델은 사용하지 않습니다.


## 실습 준비

이번 실습에서도 **Ames Housing** 데이터를 사용합니다.

### 사용할 주요 컬럼

- `GrLivArea`: 지상 생활 면적
- `OverallQual`: 주택의 전반적인 품질
- `YearBuilt`: 건축 연도
- `LotArea`: 대지 면적
- `SalePrice`: 주택 판매 가격

먼저 데이터를 불러오고, 사용할 컬럼의 자료형·결측치·기초 통계량을 확인합니다.


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("Ames_Housing.csv")

use_cols = ["GrLivArea", "OverallQual", "YearBuilt", "LotArea", "SalePrice"]
data = df[use_cols].copy()

print("데이터 크기:", data.shape)
display(data.head())


데이터 크기: (1460, 5)


,GrLivArea,OverallQual,YearBuilt,LotArea,SalePrice
0,1710,7,2003,8450,208500
1,1262,6,1976,9600,181500
2,1786,7,2001,11250,223500
3,1717,7,1915,9550,140000
4,2198,8,2000,14260,250000


In [2]:
print("[자료형]")
display(data.dtypes.to_frame("dtype"))

print("[결측치 수]")
display(data.isna().sum().to_frame("missing_count"))

print("[기초 통계량]")
display(data.describe())


[자료형]


,dtype
GrLivArea,int64
OverallQual,int64
YearBuilt,int64
LotArea,int64
SalePrice,int64


[결측치 수]


,missing_count
GrLivArea,0
OverallQual,0
YearBuilt,0
LotArea,0
SalePrice,0


[기초 통계량]


,GrLivArea,OverallQual,YearBuilt,LotArea,SalePrice
count,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000
mean,1515.463699,6.099315,1971.267808,10516.828082,180921.195890
std,525.480383,1.382997,30.202904,9981.264932,79442.502883
min,334.000000,1.000000,1872.000000,1300.000000,34900.000000
25%,1129.500000,5.000000,1954.000000,7553.500000,129975.000000
50%,1464.000000,6.000000,1973.000000,9478.500000,163000.000000
75%,1776.750000,7.000000,2000.000000,11601.500000,214000.000000
max,5642.000000,10.000000,2010.000000,215245.000000,755000.000000


---

# 필수 1. Linear Regression으로 기준 모델의 성격 확인

## 배경

교안에서 **Linear Regression**은 연속적인 숫자를 예측하는 회귀 모델이며, 각 특성에 가중치를 곱해 더하는 구조라고 배웠습니다. 또한 가중치를 확인할 수 있어 **해석이 쉬운 모델**이라는 특징이 있습니다.

Ames Housing에서 `GrLivArea`, `OverallQual`, `YearBuilt`를 이용해 `SalePrice`를 예측하는 Linear Regression을 만들어봅니다.

## 문제

### 요구사항

1. 입력 데이터 `X`는 `GrLivArea`, `OverallQual`, `YearBuilt`로 구성하고, 정답 `y`는 `SalePrice`로 지정하세요.
2. `train_test_split()`을 이용해 train과 test로 나누세요. `test_size=0.2`, `random_state=42`를 사용합니다.
3. `LinearRegression()` 모델을 만들고 train 데이터로 학습하세요.
4. 학습한 모델의 `intercept_`와 각 특성의 `coef_`를 출력하세요.
5. test 데이터의 앞 5개 행에 대해 예측값을 만들고 실제 `SalePrice`와 나란히 확인하세요.

### 확인 포인트

- `SalePrice`가 연속적인 숫자이므로 Linear Regression을 적용할 수 있는지 확인합니다.
- 계수(`coef_`)를 통해 각 입력 특성이 예측식에서 어떤 방향의 가중치를 갖는지 확인합니다.
- 이번 문제에서는 **모델 성능지표를 계산하지 않습니다.**

### Q&A

- **Q1.** Linear Regression은 회귀와 분류 중 어느 문제에 사용하는 모델인가요?
- **Q2.** Linear Regression이 해석이 쉬운 모델로 설명되는 이유는 무엇인가요?
- **Q3.** `SalePrice` 예측에 Logistic Regression을 사용하지 않는 이유는 무엇인가요?


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# TODO 1. X와 y를 만드세요.
x = data[['GrLivArea', 'OverallQual', 'YearBuilt']]
y = data['SalePrice']

# TODO 2. train / test로 분리하세요.
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=42)

# TODO 3. Linear Regression을 학습하세요.
Linear_model = LinearRegression()
Linear_model.fit(x_train,y_train)

# TODO 4. 절편과 계수를 출력하세요.
print("y절편:", Linear_model.intercept_)
print("계수  :", Linear_model.coef_)

print("\n[특성별 계수]")
for name, c in zip(x.columns, Linear_model.coef_):
    print(f"  {name:12s}: {c:>12.4f}")

# TODO 5. test 앞 5개 행의 실제값과 예측값을 비교하세요.
y_pred = Linear_model.predict(x_test)

compare = pd.DataFrame({
    "실제값": y_test.values[:5],
    "예측값": y_pred[:5]
})
compare["차이"] = compare["실제값"] - compare["예측값"]

print("\n[test 앞 5개 행 비교]")
display(compare.round(2))


y절편: -1075160.0572640228
계수  : [   60.13246816 24520.1612445    514.79999511]

[특성별 계수]
  GrLivArea   :      60.1325
  OverallQual :   24520.1612
  YearBuilt   :     514.8000

[test 앞 5개 행 비교]


,실제값,예측값,차이
0,154500,146734.78,7765.22
1,325000,305179.75,19820.25
2,115000,101276.52,13723.48
3,159000,174336.93,-15336.93
4,315500,276139.58,39360.42


In [4]:
# from sklearn.model_selection import train_test_split
# from sklearn.linear_model import LinearRegression

# # TODO 1. X와 y를 만드세요.
# # X에는 GrLivArea(지상 생활 면적), OverallQual(주택 전반 품질 점수), YearBuilt(건축연도) 컬럼 사용
# X = data[['GrLivArea', 'OverallQual', 'YearBuilt']]
# y = data['SalePrice']

# # TODO 2. train / test로 분리하세요.
# # 80 : 20으로 나눔
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # TODO 3. Linear Regression을 학습하세요.
# # 모델 객체 생성 후 fit
# linear_model = LinearRegression()
# linear_model.fit(X_train, y_train)

# # TODO 4. 절편과 계수를 출력하세요.
# # intercept_, coef_ 확인
# print("y 절편 :", linear_model.intercept_)
# coef_table = pd.DataFrame({"feature" : X.columns, "coefficient" : linear_model.coef_})
# display(coef_table)

# # TODO 5. test 앞 5개 행의 실제값과 예측값을 비교하세요.
# # test.head() 값 predict하여 확인
# sample_pred = linear_model.predict(X_test.head())
# sample_result = pd.DataFrame({
#     "actual_SalePrice": y_test.head().values,
#     "predicted_SalePrice": sample_pred}, index=X_test.head().index)
# display(sample_result)

### 필수 1 답변 작성
### 1번 표 해석
GrLivArea의 계수 60.13은 품질 및 건축연도가 같을 때 면적 1제곱피트 증가에 예측 가격이 약 60.13달러가 증가.
OverallQual의 계수 24,520은 면적 및 건축연도가 같을때 품질 점수 1점 증가에 예측 가격 약 24,520달러가 증가.
YearBuilt의 계수 514.7은 면적 및 품질이 같을 때 건축연도가 1년 뒤일수록 예측 가격이 약 514.7달러 증가

->OverallQual이 가장 큰 숫자이지만 단위 수가 다르기 때문에 가장 중요한 변수라고 결론을 내리면 안된다.

### 2번 표 해석
원본 행 인덱스 892 : 실제가격 154,500달러 예측가격 146,734, 예측- 실제 = -7,765달러
-> 실제보다 약 7,765달러 낮게 예측

원본 행 인덱스 1105 : 실제가격 325,000달러 예측가격 305,179, 예측 - 실제 = -19,820달러
-> 실제보다 약 19,820달러 낮게 예측

### 확인할 수 있는 것 : 선형 회귀 각 특성별 계수와 절편을 학습하여 가격을 예측
### 아직 확인 못한 것 : 전체 Test 성능(head()값만 테스트), 다른 모델과의 비교, 실무적합도 파악
### 최종 해석 : 주택 가격을 예측하는 선형회귀 모델 학습 결과, 새 입력 특성의 계수는 모두 양수였고 -> 다른 입력값을 고정하면 면적, 품질, 건축연도가 커질수록 예측 가격이 증가하는 구조.


- **Q1. Linear Regression의 문제 유형:**  
회귀유형

- **Q2. 해석이 쉬운 이유:**  
-> 각 특성에 적용된 가중치(계수)를 직접적으로 확인이 가능한다.

- **Q3. Logistic Regression을 사용하지 않는 이유:**
-> 로지스틱 회귀는 분류이기 때문에 현재 회귀 문제에서 다루지 않음

- **Q1.** Linear Regression은 회귀와 분류 중 어느 문제에 사용하는 모델인가요?

- **Q2.** Linear Regression이 해석이 쉬운 모델로 설명되는 이유는 무엇인가요?

- **Q3.** `SalePrice` 예측에 Logistic Regression을 사용하지 않는 이유는 무엇인가요?

---

# 필수 2. K-means와 PCA로 비지도학습 목적 구분

## 배경

교안에서 **K-means**는 정답 레이블 없이 비슷한 데이터를 K개의 군집으로 묶는 군집화 알고리즘이라고 배웠습니다. 거리로 가까운 중심을 찾기 때문에 **특성 스케일에 민감**하여 스케일링을 적용하는 것이 좋습니다.

반면 **PCA**는 많은 특성을 더 적은 수의 새로운 축으로 압축하는 **차원축소** 기법입니다.

같은 Ames Housing의 숫자형 특성을 이용해 두 기법이 결과를 어떻게 다르게 만드는지 확인합니다.

## 문제

### 요구사항

1. `GrLivArea`, `OverallQual`, `YearBuilt`, `LotArea` 네 컬럼으로 `X_unsup`을 만드세요. `SalePrice`는 사용하지 않습니다.
2. `StandardScaler()`로 `X_unsup`의 스케일을 맞추세요.
3. 스케일링한 데이터에 `KMeans(n_clusters=3, random_state=42, n_init=10)`을 적용하고 각 행의 군집 번호를 구하세요.
4. 군집별 데이터 개수를 출력하세요.
5. 원본 네 특성에 `PCA(n_components=2)`를 적용해 2개의 새로운 축으로 변환하세요.
6. PCA 적용 전과 적용 후 데이터의 크기(shape)를 출력하세요.

### 확인 포인트

- K-means에서는 정답 `y` 없이 특성 데이터만 사용합니다.
- K-means의 K는 **군집 개수**를 의미합니다.
- PCA는 데이터를 그룹으로 나누는 것이 아니라 **특성의 개수를 줄이는 것**이 목적입니다.
- 이번 실습에서 K-means의 K를 고르기 위한 Elbow/Silhouette 비교는 진행하지 않습니다.

### Q&A

- **Q1.** K-means는 지도학습과 비지도학습 중 어디에 해당하나요?
- **Q2.** K-means 전에 스케일링을 적용한 이유는 무엇인가요?
- **Q3.** K-means의 K와 KNN의 K는 각각 무엇의 개수인가요?
- **Q4.** PCA를 적용한 뒤 `(행 개수, 특성 개수)`에서 어떤 부분이 바뀌나요?


In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# TODO 1. 비지도학습에 사용할 X_unsup을 만드세요.
# GrLivArea, OverallQual, YearBuilt, LotArea(주택 면적)
X_unsup = data[['GrLivArea', 'OverallQual', 'YearBuilt', 'LotArea']]

# TODO 2. StandardScaler로 스케일을 맞추세요.
# StandardScaler(), fit_transform()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_unsup)

# TODO 3. K-means로 3개 군집을 만들고 군집 번호를 구하세요.
# n_clusters=3, random_state=42, n_init=10
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

# TODO 4. 군집별 데이터 개수를 출력하세요.
# value_counts, sort_index
cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
print(cluster_counts)

# TODO 5. PCA로 4개 특성을 2개의 새로운 축으로 변환하세요.
# PCA, fit_transform()
pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X_unsup)

# TODO 6. PCA 적용 전/후 shape를 출력하세요.
# shape()
print(X_unsup.shape)
print(X_reduced.shape)

0    819
1    637
2      4
Name: count, dtype: int64
(1460, 4)
(1460, 2)


In [6]:
X_unsup.describe()

,GrLivArea,OverallQual,YearBuilt,LotArea
count,1460.000000,1460.000000,1460.000000,1460.000000
mean,1515.463699,6.099315,1971.267808,10516.828082
std,525.480383,1.382997,30.202904,9981.264932
min,334.000000,1.000000,1872.000000,1300.000000
25%,1129.500000,5.000000,1954.000000,7553.500000
50%,1464.000000,6.000000,1973.000000,9478.500000
75%,1776.750000,7.000000,2000.000000,11601.500000
max,5642.000000,10.000000,2010.000000,215245.000000


In [6]:
print("축별 설명 비율:", pca.explained_variance_ratio_)
print("2개 축 합계   :", pca.explained_variance_ratio_.sum())

축별 설명 비율: [0.99741871 0.00257252]
2개 축 합계   : 0.999991233784838


### 필수 2 답변 작성

### 군집별 주택 수
### 군집번호(0) : 주택 수 819개, 가장 큰 군집
### 군집번호(1) : 주택 수 637개, 두번째로 큰 군집
### 군집번호(2) : 주택 수 4개, 가장 작은 군집

#### 군집 크기가 다른 두 집단에 비해 굉장히 작은 4는 오류라고 봐야하는가? 아니면 잘못된 결과라고 해석해야하는건가?

#### 전체 0.3% 정도의 분포를 보이는 군집번호 2는 이상치나 잘못된 부분은 아니다.

### PCA (1460, 4) -> (1460, 2)
### 관측대상 : 기존(주택 1,460채), PCA 후 (주택 1,460채)
### 한 주택을 표현하는 특성 X : 기존 (4개) , PCA 후 (2개)
### 열의 의미 : 기존(면적, 품질, 건축연도, 대지면적), PCA 후 (원래 특성들을 조합한 새로운 축)

#### Y를 설명하려던 변수 4개가 2개로 줄었는데 정보 손실이 발생한 것 아닌가? 그 이유는?
#### -> PCA는 변수 4개에 대한 유사한 변동성을 가진 데이터를 묶으려는 과정
#### -> 4개의 변수를 2개로 줄이면 일부 변동성을 충분히 반영하지 못할 수 있지만
#### -> 이 부분이 반드시 정보의 50%를 손실했다고 볼 수 없다.
#### (변수의 갯수가 줄어들면서 정보 손실이 발생하는 것은 맞으나 그게 줄어든 숫자만큼의 %로 손실되는 것은 아니다. 정확한 정보 손실률을 보려면 모델링으로 측정가능)

- **Q1. K-means의 학습 방식:**  
-> 비지도 학습

- **Q2. K-means 전에 스케일링이 필요한 이유:**  
-> 원본 통계량에서 품질은 1~10점, 대지면적은 1,300~ 215,000정도로 
-> 원본 값에서 좌표 거리를 계산하면 숫자 변동이 매우 커지는 경우가 있다.
-> 표준화는 이러한 변동 규모와 단위를 줄여 영향 차이를 줄일 수 있다.
-> 단, 표준화가 이상치를 제거하거나, 중요도를 동일 시 하게 만든다는 의미는 아니다.

- **Q3. K-means의 K / KNN의 K 차이:**  
-> K-Means는 군집의 개수 / KNN의 K는 이웃의 개수

- **Q4. PCA 적용 전후 shape 변화:**
-> 행 개수는 유지, Y를 설명하는 X변수가 4개에서 2개로 줄었다. 즉, 특성(열) 개수가 4개에서 2개로 줄었다


---

# 과제 1. Ames Housing 상황에 맞는 모델 후보 선택

## 배경

이번 강의의 핵심은 모델을 복잡하게 튜닝하는 것이 아니라, **데이터의 목적과 특성을 보고 어떤 모델부터 후보로 볼지 판단하는 것**입니다.

아래 상황은 모두 이번 강의에서 학습한 Linear Regression, Logistic Regression, KNN, Decision Tree, K-means, PCA의 특징만으로 판단할 수 있습니다.

## 문제

각 상황에서 가장 먼저 고려할 모델 또는 기법을 하나 선택하고, **교안에서 배운 선택 기준을 근거로 이유를 한 문장으로 작성하세요.**

### 요구사항

1. `SalePrice`처럼 연속적인 숫자를 예측하면서, 각 특성의 가중치를 확인해 예측 근거도 쉽게 설명하고 싶습니다. 어떤 모델을 먼저 고려해야 하나요?
2. 주택 특성 사이의 관계가 단순한 직선 형태가 아니며, 사람이 이해할 수 있는 `예/아니오` 규칙 형태로 나누고 싶습니다. 어떤 모델을 먼저 고려해야 하나요?
3. 거리 기반으로 주변의 비슷한 주택을 참고하여 예측하려고 합니다. 어떤 모델을 고려할 수 있으며, 사용 전에 무엇을 확인해야 하나요?
4. `SalePrice` 같은 정답 레이블을 사용하지 않고 주택을 비슷한 특성끼리 3개 그룹으로 묶고 싶습니다. 어떤 기법을 사용해야 하나요?
5. 숫자형 특성이 많아 분석하기 어렵기 때문에 핵심 정보를 유지하면서 2개의 새로운 축으로 줄이고 싶습니다. 어떤 기법을 사용해야 하나요?

### 확인 포인트

- 모델 이름만 쓰지 않고 **왜 그 모델을 선택했는지** 교안의 특성과 연결합니다.
- 성능지표 비교, 교차검증, GridSearch 등의 방법은 사용하지 않습니다.
- 과제는 필수 문제에서 학습한 **모델의 성격과 선택 기준을 다시 적용하는 수준**입니다.


### 과제 1 답변 작성

1. **연속값 예측 + 해석 중요:**  
-> Linear Regression. 연속적인 숫자를 예측하는 회귀 모델이고, 특성마다 가중치를 곱해 더하는 구조다. 계수를 읽어 예측 근거를 설명할 수 있다.

2. **비선형 관계 + 규칙 해석:**  
-> Decision Tree. 직선이 아닌 관계도 다룰 수 있고, 예/아니오 질문을 이어 붙이는 규칙 기반이다. 사람이 분할 과정을 그대로 따라 읽을 수 있다.

3. **가까운 이웃 기반 예측:**  
-> KNN. 주변에 비슷한 데이터를 참고하는 거리 기반 모델이므로, 사용 전에 스케일링이 되어 있는지 확인이 필요하다.

4. **정답 없이 3개 그룹으로 군집화:**  
->  K-means. 정답 레이블 없이 비슷한 데이터끼리 묶는 비지도학습이고, K가 군집 개수이므로 n_clusters=3으로 지정한다.

5. **많은 특성을 2개 축으로 압축:**
-> PCA. 그룹으로 나누는 것이 아닌 특성 개수를 줄이는 차원축소 기법이다.

---

# 실습 마무리

아래 내용을 한 문장씩 정리해보세요.

1. **Linear Regression은 어떤 상황에서 1차 후보가 될 수 있는가?**
-> 회귀 문제, 각 특성의 가중치를 통해 결과를 쉽게 설명하고 빠르게 결과를 받을 때 1차 후보로 고려.

2. **KNN / Decision Tree는 데이터 특성에 따라 어떤 차이가 있는가?**
-> KNN은 거리 기반이므로 스케일링에 민감하고, Decision Tree(결정트리) 는 규칙 기반으로 데이터를 나누기 때문에 스케일링이 필요하지 않음

3. **K-means와 PCA의 목적은 어떻게 다른가?**
-> K-Means는 데이터를 비슷한 것 끼리 군집으로 묶는 것
-> PCA 는 많은 특성(X)을 더 작은 새로운 축으로 줄이는 것

4. **처음부터 복잡한 모델을 선택하기보다 후보를 좁힐 때 어떤 기준을 볼 수 있는가?**
-> 문제의 목적(회귀?/분류?), 비선형성인지 선형상인지?, 스케일링이 필요한지? 차원 축소가 필요한지?
